# 01. OpenAIEmbeddings

In [1]:
from langchain_teddynote import logging
from dotenv import load_dotenv

load_dotenv()
logging.langsmith('CH08-Embeddings')

LangSmith 추적을 시작합니다.
[프로젝트명]
CH08-Embeddings


In [2]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

In [3]:
text = '임베딩 테스트를 하기 위한 샘플 문장입니다.'

In [4]:
query_result = embeddings.embed_query(text)

In [5]:
len(query_result)

1536

In [6]:
query_result[:5]

[-0.007762908935546875,
 0.036712646484375,
 0.01953125,
 -0.0196990966796875,
 0.0172119140625]

In [7]:
doc_result = embeddings.embed_documents(
    [text, text, text, text]
)

In [8]:
doc_result[0][:5]

[-0.00775146484375,
 0.036712646484375,
 0.01953125,
 -0.0196990966796875,
 0.0172119140625]

In [9]:
len(doc_result[0])

1536

In [10]:
embeddings_1024 = OpenAIEmbeddings(model='text-embedding-3-small', dimensions=1024)

len(embeddings_1024.embed_documents([text])[0])

1024

In [11]:
from sklearn.metrics.pairwise import cosine_similarity

sentence1 = '안녕하세요? 반갑습니다.'
sentence2 = '안녕하세요? 반갑습니다!'
sentence3 = '안녕하세요? 만나서 반가워요.'
sentence4 = 'Hi, nice to meet you.'
sentence5 = 'I like to eat apples.'

sentences = [sentence1, sentence2, sentence3, sentence4, sentence5]
embedded_sentences = embeddings_1024.embed_documents(sentences)

In [12]:
def similarity(a, b):
    return cosine_similarity([a], [b])[0][0]

In [13]:
for i, sentence in enumerate(embedded_sentences):
    for j, other_sentence in enumerate(embedded_sentences):
        if i < j:
            print(
                f"[유사도 {similarity(sentence, other_sentence):.4f}] {sentences[i]} \t <=====> \t {sentences[j]}"
            )

[유사도 0.9644] 안녕하세요? 반갑습니다. 	 <=====> 	 안녕하세요? 반갑습니다!
[유사도 0.8423] 안녕하세요? 반갑습니다. 	 <=====> 	 안녕하세요? 만나서 반가워요.
[유사도 0.5043] 안녕하세요? 반갑습니다. 	 <=====> 	 Hi, nice to meet you.
[유사도 0.1363] 안녕하세요? 반갑습니다. 	 <=====> 	 I like to eat apples.
[유사도 0.8185] 안녕하세요? 반갑습니다! 	 <=====> 	 안녕하세요? 만나서 반가워요.
[유사도 0.4791] 안녕하세요? 반갑습니다! 	 <=====> 	 Hi, nice to meet you.
[유사도 0.1320] 안녕하세요? 반갑습니다! 	 <=====> 	 I like to eat apples.
[유사도 0.5164] 안녕하세요? 만나서 반가워요. 	 <=====> 	 Hi, nice to meet you.
[유사도 0.1458] 안녕하세요? 만나서 반가워요. 	 <=====> 	 I like to eat apples.
[유사도 0.2249] Hi, nice to meet you. 	 <=====> 	 I like to eat apples.


# 02. CacheBackedEmbeddings

In [5]:
from langchain.storage import LocalFileStore
from langchain.embeddings import CacheBackedEmbeddings
from langchain_community.vectorstores.faiss import FAISS
embedding = OpenAIEmbeddings()

store = LocalFileStore('./cache/')

In [6]:
cached_embedder = CacheBackedEmbeddings.from_bytes_store(
    underlying_embeddings=embedding,
    document_embedding_cache=store,
    namespace=embedding.model,
)

In [7]:
list(store.yield_keys())

[]

In [9]:
from langchain.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter

raw_documents = TextLoader('../langchain-kr/08-Embeddings/data/appendix-keywords.txt', encoding='utf-8').load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
documents = text_splitter.split_documents(raw_documents)

In [10]:
%time db = FAISS.from_documents(documents, cached_embedder)

CPU times: total: 891 ms
Wall time: 4.2 s


In [11]:
%time db2 = FAISS.from_documents(documents, cached_embedder)

CPU times: total: 31.2 ms
Wall time: 41.9 ms


In [12]:
from langchain.storage import InMemoryByteStore

store = InMemoryByteStore()

cached_embedder = CacheBackedEmbeddings.from_bytes_store(
    embedding, store, namespace=embedding.model
)

# 03. HuggingFaceEmbeddings

In [13]:
import os
import warnings

warnings.filterwarnings('ignore')

os.environ['HF_HOME'] = './cache/'

In [6]:
texts = [
    '안녕, 만나서 반가워.',
    'Langchain simplifies the process of building applications with large language models',
    '랭체인 한국어 튜토리얼은 LangChain의 공식 문서, cookbook 및 다양한 실용 예제를 바탕으로 하여 사용자가 LangChain을 더 쉽고 효과적으로 활용할 수 있도록 구성되어 있습니다.',
    'LangChain은 초거대 언어모델로 애플리케이션을 구축하는 과정을 단순화합니다.',
    'Retrieval-Augmented Generation (RAG) is an effective technique for improving AI responses.'
]

In [22]:
from dotenv import load_dotenv

load_dotenv()

True

In [31]:
from langchain_huggingface.embeddings import HuggingFaceEndpointEmbeddings

model_name = 'intfloat/multilingual-e5-large-instruction'

hf_embeddings = HuggingFaceEndpointEmbeddings(
    model=model_name,
    task='feature-extraction',
    huggingfacehub_api_token=os.environ['HUGGINGFACEHUB_API_TOKEN']
)

In [32]:
%%time
embedded_documents = hf_embeddings.embed_documents(texts)

CPU times: total: 0 ns
Wall time: 237 ms


HfHubHTTPError: 404 Client Error: Not Found for url: https://router.huggingface.co/hf-inference/pipeline/feature-extraction/intfloat/multilingual-e5-large-instruction (Request ID: Root=1-69c7338e-4ee263c0627e8f612c46d0fd;77682c87-a769-4cd5-8797-39091265e94f)

In [33]:
print('[HuggingFace Endpoint Embedding]')
print(f'Model: \t\t{model_name}')
print(f'Dimension: \t{len(embedded_documents[0])}')

[HuggingFace Endpoint Embedding]
Model: 		intfloat/multilingual-e5-large-instruction


NameError: name 'embedded_documents' is not defined

In [34]:
embedded_query = hf_embeddings.embed_query('LangChain에 대해서 알려주세요.')
embedded_query

HfHubHTTPError: 404 Client Error: Not Found for url: https://router.huggingface.co/hf-inference/pipeline/feature-extraction/intfloat/multilingual-e5-large-instruction (Request ID: Root=1-69c7339e-6dc232372ebec7142ed50ee8;e103aa92-6c98-4245-aceb-065570699379)

In [2]:
import numpy as np

np.array(embedded_query) @ np.array(embedded_documents).T

NameError: name 'embedded_query' is not defined

In [3]:
sorted_idx = (np.array(embedded_query) @ np.array(embedded_documents).T).argsort()[::-1]
sorted_idx

NameError: name 'embedded_query' is not defined

In [4]:
print('[Query] Langchain 에 대해서 알려주세요.\n==============================')
for i, idx in enumerate(sorted_idx):
    print(f'[{i}] {texts[idx]}')
    print()

[Query] Langchain 에 대해서 알려주세요.


NameError: name 'sorted_idx' is not defined

In [5]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

model_name = 'intfloat/multilingual-e5-large-instruct'

hf_embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs={'device': 'cuda'},
    encode_kwargs={'normalize_embeddings': True}
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/128 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_xlm-roberta_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

In [7]:
%time
embedded_documents = hf_embeddings.embed_documents(texts)

CPU times: total: 0 ns
Wall time: 0 ns


In [8]:
print(f'Model: \t\t{model_name}')
print(f'Dimension: \t{len(embedded_documents[0])}')

Model: 		intfloat/multilingual-e5-large-instruct
Dimension: 	1024


In [9]:
model_name = 'BAAI/bge-m3'
model_kwargs = {'device': 'cuda'}
encode_kwargs = {'normalize_embeddings': True}
hf_embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

%time
embedded_documents = hf_embeddings.embed_documents(texts)

print(f'Model: \t\t{model_name}')
print(f'Dimension: \t{len(embedded_documents[0])}')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

CPU times: total: 0 ns
Wall time: 0 ns
Model: 		BAAI/bge-m3
Dimension: 	1024


In [10]:
import numpy as np

embedded_query = hf_embeddings.embed_query('LangChain에 대해서 알려주세요.')
embedded_documents = hf_embeddings.embed_documents(texts)

np.array(embedded_query) @ np.array(embedded_documents).T

sorted_idx = (np.array(embedded_query) @ np.array(embedded_documents).T).argsort()[::-1]

print('[Query] LangChain에 대해서 알려주세요.\n==========================')
for i, idx in enumerate(sorted_idx):
    print(f'[{i}] {texts[idx]}')
    print()

[Query] LangChain에 대해서 알려주세요.
[0] LangChain은 초거대 언어모델로 애플리케이션을 구축하는 과정을 단순화합니다.

[1] Langchain simplifies the process of building applications with large language models

[2] 랭체인 한국어 튜토리얼은 LangChain의 공식 문서, cookbook 및 다양한 실용 예제를 바탕으로 하여 사용자가 LangChain을 더 쉽고 효과적으로 활용할 수 있도록 구성되어 있습니다.

[3] 안녕, 만나서 반가워.

[4] Retrieval-Augmented Generation (RAG) is an effective technique for improving AI responses.



In [13]:
from FlagEmbedding import BGEM3FlagModel

model_name = 'BAAI/bge-m3'
bge_embeddings = BGEM3FlagModel(
    model_name, use_fp16=True
)

bge_embedded = bge_embeddings.encode(
    texts,
    batch_size=12,
    max_length=8192,
)['dense_vecs']

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

nqa.jpg:   0%|          | 0.00/158k [00:00<?, ?B/s]

mkqa.jpg:   0%|          | 0.00/608k [00:00<?, ?B/s]

.DS_Store:   0%|          | 0.00/6.15k [00:00<?, ?B/s]

miracl.jpg:   0%|          | 0.00/576k [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

model.onnx:   0%|          | 0.00/725k [00:00<?, ?B/s]

model.onnx_data:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

sparse_linear.pt:   0%|          | 0.00/3.52k [00:00<?, ?B/s]

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


In [14]:
bge_embedded.shape

(5, 1024)

In [15]:
from FlagEmbedding import BGEM3FlagModel

bge_flagmodel = BGEM3FlagModel(
    'BAAI/bge-m3', use_fp16=True
)
bge_encoded = bge_flagmodel.encode(texts, return_dense=True)

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


In [16]:
bge_encoded['dense_vecs'].shape

(5, 1024)

In [18]:
bge_flagmodel = BGEM3FlagModel(
    'BAAI/bge-m3', use_fp16=True
)
bge_encoded = bge_flagmodel.encode(texts, return_sparse=True)

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


In [19]:
lexical_scores1 = bge_flagmodel.compute_lexical_matching_score(
    bge_encoded['lexical_weights'][0], bge_encoded['lexical_weights'][0]
)
lexical_scores2 = bge_flagmodel.compute_lexical_matching_score(
    bge_encoded['lexical_weights'][0], bge_encoded['lexical_weights'][1]
)

print(lexical_scores1)
print(lexical_scores2)

0.3018
0


In [20]:
bge_flagmodel = BGEM3FlagModel(
    'BAAI/bge-m3', use_fp16=True
)
bge_encoded = bge_flagmodel.encode(texts, return_colbert_vecs=True)

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


In [21]:
colbert_scores1 = bge_flagmodel.colbert_score(
    bge_encoded['colbert_vecs'][0], bge_encoded['colbert_vecs'][0]
)
colbert_scores2 = bge_flagmodel.colbert_score(
    bge_encoded['colbert_vecs'][0], bge_encoded['colbert_vecs'][1]
)

print(colbert_scores1)
print(colbert_scores2)

tensor(1.)
tensor(0.3645)


# 04. UpstageEmbeddings

In [22]:
texts

['안녕, 만나서 반가워.',
 'Langchain simplifies the process of building applications with large language models',
 '랭체인 한국어 튜토리얼은 LangChain의 공식 문서, cookbook 및 다양한 실용 예제를 바탕으로 하여 사용자가 LangChain을 더 쉽고 효과적으로 활용할 수 있도록 구성되어 있습니다.',
 'LangChain은 초거대 언어모델로 애플리케이션을 구축하는 과정을 단순화합니다.',
 'Retrieval-Augmented Generation (RAG) is an effective technique for improving AI responses.']

In [23]:
from langchain_upstage import UpstageEmbeddings

query_embeddings = UpstageEmbeddings(model='solar-embedding-1-large-query')
passage_embeddings = UpstageEmbeddings(model='solar-embedding-1-large-passage')

In [24]:
embedded_query = query_embeddings.embed_query('LangChain에 대해서 상세히 알려주세요.')
len(embedded_query)

AuthenticationError: Error code: 401 - {'error': {'message': 'API key suspended due to insufficient credit. Register your payment method at https://console.upstage.ai/billing to continue.', 'type': 'invalid_request_error', 'param': '', 'code': 'api_key_is_not_allowed'}}

In [1]:
embedded_documents = passage_embeddings.embed_documents(texts)

NameError: name 'passage_embeddings' is not defined

In [2]:
import numpy as np

similarity = np.array(embedded_query) @ np.array(embedded_documents).T

sorted_idx = (np.array(embedded_query) @ np.array(embedded_documents).T).argsort()[::-1]

print('[Query] LangChain에 대해서 알려주세요.\n==================================')
for i, idx in enumerate(sorted_idx):
    print(f'[{i}] 유사도: {similarity[idx]:.3f} | {texts[idx]}')

NameError: name 'embedded_query' is not defined

# 05. OllamaEmbeddings

In [1]:
import os
os.environ["OLLAMA_HOST"] = "127.0.0.1"
# !ollama pull nomic-embed-text

In [2]:
texts = [
    '안녕, 만나서 반가워.',
    'LangChain simplifies the process of building applications with large language models',
    '랭체인 한국어 튜토리얼은 LangChain의 공식 문서, cookbook 및 다양한 실용 예제를 바탕으로 하여 사용자가 LangChain을 더 쉽고 효과적으로 활용할 수 있도록 구성되어 있습니다. ',
    'LangChain은 초거대 언어모델로 애플리케이션을 구축하는 과정을 단순화합니다.',
    'Retrieval-Augmented Generation (RAG) is an effective technique for improving AI responses.'
]

In [3]:
from langchain_community.embeddings import OllamaEmbeddings
from dotenv import load_dotenv

load_dotenv()

ollama_embeddings = OllamaEmbeddings(
    model='nomic-embed-text'
)

C:\Users\USER\AppData\Local\Temp\ipykernel_13520\1596462270.py:6: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaEmbeddings``.
  ollama_embeddings = OllamaEmbeddings(


In [4]:
embedded_query = ollama_embeddings.embed_query('LangChain에 대해서 상세히 알려주세요.')
len(embedded_query)

768

In [5]:
embedded_documents = ollama_embeddings.embed_documents(texts)

In [7]:
import numpy as np

similarity = np.array(embedded_query) @ np.array(embedded_documents).T
sorted_idx = (np.array(embedded_query) @ np.array(embedded_documents).T).argsort()[::-1]

print(f'[Query] LangChain에 대해서 알려주세요.\n========================')
for i, idx in enumerate(sorted_idx):
    print(f'[{i}] 유사도: {similarity[idx]:.3f} | {texts[idx]}')

[Query] LangChain에 대해서 알려주세요.
[0] 유사도: 410.372 | 안녕, 만나서 반가워.
[1] 유사도: 410.237 | LangChain은 초거대 언어모델로 애플리케이션을 구축하는 과정을 단순화합니다.
[2] 유사도: 320.459 | 랭체인 한국어 튜토리얼은 LangChain의 공식 문서, cookbook 및 다양한 실용 예제를 바탕으로 하여 사용자가 LangChain을 더 쉽고 효과적으로 활용할 수 있도록 구성되어 있습니다. 
[3] 유사도: 312.683 | Retrieval-Augmented Generation (RAG) is an effective technique for improving AI responses.
[4] 유사도: 234.480 | LangChain simplifies the process of building applications with large language models
